# Hugging Face Transformers Pipeline - NLP

Kaggle Notebook에서 바로 실행 가능한 Hugging Face `transformers.pipeline` 기반 NLP 예제입니다.

포함 예제:
1. 감정 분석: `sentiment-analysis`
2. 질의 응답: `question-answering`
3. 요약: `summarization`
4. 번역: `translation`

In [ ]:
# Kaggle Notebook에서 필요한 라이브러리 설치
# transformers 5.x 계열에서는 일부 pipeline task 이름이 바뀌어 summarization이 동작하지 않을 수 있습니다.
# 이 예제는 sentiment-analysis, question-answering, summarization, translation이 모두 동작하는 4.x 버전으로 고정합니다.
# 이미 transformers를 import한 뒤 이 셀을 실행했다면 Kaggle 메뉴에서 Runtime -> Restart session 후 처음부터 다시 실행하세요.
!pip install -q -U "transformers==4.48.3" sentencepiece sacremoses

In [ ]:
# Hugging Face Transformers의 고수준 추론 API인 pipeline 함수를 불러옵니다.
import transformers
from transformers import pipeline

print("transformers version:", transformers.__version__)

## 1. 감정 분석 Sentiment Analysis

입력 문장이 긍정인지 부정인지 분류합니다.

In [ ]:
# 감정 분석 pipeline을 생성합니다.
# model을 명시하면 Kaggle에서도 동일한 모델로 재현 가능하게 실행됩니다.
sentiment_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

# 분석할 영어 문장 목록입니다.
sentences = [
    "I love using Hugging Face pipelines because they are simple and powerful.",
    "The movie was too long and the story was boring.",
]

# pipeline에 문장 리스트를 넣으면 각 문장에 대한 예측 결과가 반환됩니다.
sentiment_results = sentiment_classifier(sentences)

for sentence, result in zip(sentences, sentiment_results):
    print("Text:", sentence)
    print("Label:", result["label"])
    print("Score:", round(result["score"], 4))
    print("-" * 80)

## 2. 질의 응답 Question Answering

주어진 문맥(context)을 읽고 질문(question)에 대한 답을 찾습니다.

In [ ]:
# 질의 응답 pipeline을 생성합니다.
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
)

# 모델이 답을 찾을 문맥입니다.
context = """
Hugging Face is a company and open-source community known for the Transformers library.
The Transformers library provides thousands of pretrained models for natural language processing,
computer vision, audio, and multimodal machine learning tasks.
The pipeline function offers a simple way to run inference without writing model-specific code.
"""

# 문맥에서 답을 찾을 질문입니다.
question = "What does the pipeline function provide?"

# question과 context를 함께 전달하면 답변, 신뢰도, 위치 정보가 반환됩니다.
qa_result = qa_pipeline(question=question, context=context)

print("Question:", question)
print("Answer:", qa_result["answer"])
print("Score:", round(qa_result["score"], 4))
print("Start:", qa_result["start"])
print("End:", qa_result["end"])

## 3. 요약 Summarization

긴 영어 문서를 짧은 문장으로 요약합니다.

In [ ]:
# 요약 pipeline을 생성합니다.
# t5-small은 비교적 가벼워 Kaggle CPU 환경에서도 테스트하기 좋습니다.
summarizer = pipeline(
    "summarization",
    model="t5-small",
)

# 요약할 긴 영어 문장입니다.
article = """
The Hugging Face Transformers library makes it easier for developers and researchers to use
state-of-the-art machine learning models. Instead of manually downloading model weights,
building tokenizers, and writing task-specific inference code, users can rely on pipelines
to perform common tasks with only a few lines of Python. This is especially useful in
educational environments such as Kaggle Notebooks, where users often want to experiment
quickly with pretrained models for text classification, question answering, summarization,
translation, image classification, speech recognition, and many other tasks.
"""

# max_length와 min_length로 요약문의 길이를 조절합니다.
summary_result = summarizer(
    article,
    max_length=60,
    min_length=20,
    do_sample=False,
)

print("Original text:")
print(article.strip())
print("\nSummary:")
print(summary_result[0]["summary_text"])

## 4. 번역 Translation

영어 문장을 프랑스어로 번역합니다.

In [ ]:
# 번역 pipeline을 생성합니다.
# translation_en_to_fr는 영어를 프랑스어로 번역하는 task 이름입니다.
# Helsinki-NLP/opus-mt-en-fr 모델을 명시해서 재현 가능하게 실행합니다.
translator = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr",
)

# 번역할 영어 문장입니다.
text_to_translate = "Hugging Face pipelines make it easy to use pretrained models in Kaggle Notebooks."

# pipeline에 문장을 전달하면 번역 결과가 반환됩니다.
translation_result = translator(text_to_translate, max_length=80)

print("Original:", text_to_translate)
print("Translation:", translation_result[0]["translation_text"])

## 5. 전체 예제 한 번에 실행하기

위에서 만든 pipeline 객체를 재사용해 주요 NLP 작업 결과를 한 번에 확인합니다.

In [ ]:
# 이미 생성한 pipeline 객체들을 재사용합니다.
demo_text = "This notebook is very helpful for learning NLP pipelines."
demo_question = "What tasks does the Transformers library support?"
demo_translation_text = "Machine learning models can solve many language tasks."

print("[Sentiment Analysis]")
print(sentiment_classifier(demo_text))

print("\n[Question Answering]")
print(qa_pipeline(question=demo_question, context=context))

print("\n[Summarization]")
print(summarizer(article, max_length=50, min_length=20, do_sample=False)[0]["summary_text"])

print("\n[Translation]")
print(translator(demo_translation_text, max_length=80)[0]["translation_text"])